# 🚀 AutoML Pilot Pro - Advanced Google Colab Training
This notebook is part of the AutoML Pilot Pro ecosystem. It allows you to run high-performance training jobs on Google Colab's hardware (including GPUs/TPUs) with automated professional reporting and one-click model downloads.

In [ ]:
# @title 📦 1. Environment Setup
!pip install -q pycaret[full] ydata-profiling pandas jinja2 shap
print('✅ Environment ready!')

In [ ]:
# @title 📂 2. Data Ingestion
import pandas as pd
import os
from google.colab import files

if not os.path.exists('dataset.csv'):
    print("Please upload your CSV file:")
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    os.rename(filename, 'dataset.csv')

df = pd.read_csv('dataset.csv')
print(f'✅ Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns')

In [ ]:
# @title 🔍 3. Automated EDA (Optional)
run_eda = True # @param {type:"boolean"}
if run_eda:
    from ydata_profiling import ProfileReport
    profile = ProfileReport(df, title="AutoML Pilot EDA Report", explorative=True, minimal=True)
    profile.to_notebook_iframe()

In [ ]:
# @title 🤖 4. AutoML Training Pipeline
from pycaret.classification import setup as cls_setup, compare_models as cls_compare, pull as cls_pull, save_model as cls_save, finalize_model as cls_finalize, interpret_model as cls_interpret
from pycaret.regression import setup as reg_setup, compare_models as reg_compare, pull as reg_pull, save_model as reg_save, finalize_model as reg_finalize, interpret_model as reg_interpret

target_column = 'target' # @param {type:"string"}
task_type = 'classification' # @param ["classification", "regression"]

if task_type == 'classification':
    s = cls_setup(data=df, target=target_column, session_id=123, verbose=False, html=False)
    best_model = cls_compare()
    leaderboard = cls_pull()
    final_model = cls_finalize(best_model)
    cls_save(final_model, 'best_model')
else:
    s = reg_setup(data=df, target=target_column, session_id=123, verbose=False, html=False)
    best_model = reg_compare()
    leaderboard = reg_pull()
    final_model = reg_finalize(best_model)
    reg_save(final_model, 'best_model')

print("\n🏆 Model Leaderboard:")
display(leaderboard.head())
print(f"\n✅ Best Model Finalized: {best_model}")

In [ ]:
# @title 📧 5. Professional HTML Email Reporting
import smtplib, ssl
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from jinja2 import Template

recipient_email = '' # @param {type:"string"}
sender_email = '' # @param {type:"string"}
sender_password = '' # @param {type:"string"}

if recipient_email and sender_email and sender_password:
    msg = MIMEMultipart("alternative")
    msg["Subject"] = f"🚀 AutoML Pilot Pro - {task_type.capitalize()} Report (Colab)"
    msg["From"] = f"AutoML Pilot <{sender_email}>"
    msg["To"] = recipient_email

    template_str = """
    <html>
    <body style="font-family: Arial, sans-serif; color: #333;">
        <div style="max-width: 600px; margin: auto; border: 1px solid #ddd; padding: 20px; border-radius: 10px;">
            <h2 style="color: #4CAF50; text-align: center;">🚀 AutoML Training Report</h2>
            <p>Your model has been trained successfully on Google Colab hardware.</p>
            
            <div style="background: #f9f9f9; padding: 15px; border-radius: 5px; margin: 20px 0;">
                <h3 style="margin-top: 0;">📊 Performance Summary</h3>
                {{ table_html | safe }}
            </div>

            <p style="font-size: 0.9em; color: #666; text-align: center;">
                Generated by AutoML Pilot Pro ecosystem.<br>
                &copy; 2024 AutoML Pilot AI Labs
            </p>
        </div>
    </body>
    </html>
    """
    
    table_html = leaderboard.head().to_html(classes='table table-striped', border=0, justify='center')
    html_body = Template(template_str).render(table_html=table_html)
    
    msg.attach(MIMEText(html_body, "html"))
    context = ssl.create_default_context()
    try:
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=context) as server:
            server.login(sender_email, sender_password)
            server.sendmail(sender_email, recipient_email, msg.as_string())
        print('✅ Professional email report sent!')
    except Exception as e:
        print(f'❌ Email failed: {e}')
else:
    print('ℹ️ Email configuration missing. Skipping reporting step.')

In [ ]:
# @title 💾 6. Export Trained Model
from google.colab import files
if os.path.exists('best_model.pkl'):
    print("Downloading your trained model (.pkl)...")
    files.download('best_model.pkl')
    print('✅ Model export triggered!')
else:
    print('❌ Model file not found.')